# CSCI 3151 — Foundations of Machine Learning
## M52 — Attention mechanisms & self-attention (mini-lab)

In this in-class mini-lab, you will work through **two linked attention examples**:

1. a tiny, hand-computable **query–key–value** example on a short sentence, and  
2. a small synthetic **long-range sequence** task where the label depends on the relationship between the **first** and **last** token.

The first example lets you see the mechanics of **scores, softmax weights, and context vectors** directly. The second makes the architectural story concrete: a small **self-attention** model can compare distant positions more directly than a recurrent baseline such as a **GRU**.

Although the second DGP is deliberately synthetic, it isolates a real modelling issue: **how should a model retrieve information from far away in a sequence?** A practical risk in real systems is that people may over-trust attention plots as if they were faithful explanations. In this notebook, treat attention maps as **diagnostics**, not proof of causal reasoning.

**This mini-lab reinforces…**

- **Queries, keys, values, scores, softmax weights, and context vectors**.
- Why changing the **query** changes what is retrieved.
- How **self-attention** uses explicit **positional information**.
- How to compare **self-attention** and a **GRU** on a controlled long-range dependency task.
- Why attention visualizations can be useful **without** being complete explanations.

In [ ]:
# Setup: imports, plotting defaults, and reproducibility

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 3151
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

def stable_softmax(x):
    x = np.asarray(x, dtype=float)
    z = x - x.max()
    e = np.exp(z)
    return e / e.sum()

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def accuracy_on_loader(model, loader, device=device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.numel()
    return correct / total

def train_classifier(model, train_loader, val_loader, epochs=8, lr=3e-3, device=device):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        n_seen = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            running_loss += loss.item() * xb.size(0)
            n_seen += xb.size(0)

        history.append({
            "epoch": epoch,
            "train_loss": running_loss / n_seen,
            "val_acc": accuracy_on_loader(model, val_loader, device=device),
        })

    return pd.DataFrame(history)

print("Torch version:", torch.__version__)

## Core task 1 — Compute and interpret toy QKV attention

We start with the same kind of tiny worked example used in the module.

Sentence:

> **the animal did not move**

We will use a very small 2D embedding for each token and then ask two different questions:

- **Query A:** “which tokens matter for deciding whether an action happened?”
- **Query B:** “which tokens matter for deciding whether negation is present?”

As you run the next cells, focus on three ideas:

1. raw **scores** measure compatibility between the query and each key,
2. **softmax** turns those scores into a probability-like weighting, and
3. the **context vector** is a weighted blend of values, not a hard pointer to one token.

In [ ]:
# Toy token embeddings, keys, values, and two different queries

tokens = ["the", "animal", "did", "not", "move"]

# Small 2D toy embeddings chosen for geometric clarity
X = np.array([
    [ 0.2, 0.1],   # the
    [ 1.0, 0.8],   # animal
    [ 0.3, 0.2],   # did
    [-0.2, 1.3],   # not
    [ 1.1, 0.7],   # move
], dtype=float)

K = X.copy()   # keys
V = X.copy()   # values

queries = {
    "action happened?": np.array([ 1.0, 0.9]),
    "negation present?": np.array([-0.2, 1.4]),
}

rows = []
contexts = {}
weights_by_query = {}

for name, q in queries.items():
    scores = K @ q
    weights = stable_softmax(scores)
    context = weights @ V
    weights_by_query[name] = weights
    contexts[name] = context

    for tok, score, alpha in zip(tokens, scores, weights):
        rows.append({
            "query": name,
            "token": tok,
            "score": round(float(score), 3),
            "alpha": round(float(alpha), 3),
        })

attn_table = pd.DataFrame(rows)
print("Attention weights for both queries:")
display(attn_table.pivot(index="token", columns="query", values="alpha"))

print("\nContext vectors:")
for name, c in contexts.items():
    print(f"{name:>20} -> {np.round(c, 3)}")

In [ ]:
# Visualize the geometry and the attention weights

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
query_items = list(queries.items())

for row_idx, (name, q) in enumerate(query_items):
    weights = weights_by_query[name]

    # Left panel: token/key vectors in 2D plus the query
    ax = axes[row_idx, 0]
    for tok, vec in zip(tokens, X):
        ax.annotate("", xy=vec, xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2))
        ax.text(vec[0] + 0.03, vec[1] + 0.03, tok, fontsize=10, fontweight="bold")

    ax.annotate("", xy=q, xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2.5, linestyle="dashed"))
    ax.text(q[0] + 0.03, q[1] + 0.03, "query", fontsize=10, fontweight="bold")
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(0, color="gray", lw=0.8)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.3, 1.7)
    ax.set_title(f"2D geometry: {name}")
    ax.set_xlabel("dim 0")
    ax.set_ylabel("dim 1")

    # Right panel: bar chart of the resulting attention weights
    ax2 = axes[row_idx, 1]
    ax2.bar(tokens, weights)
    ax2.axhline(1 / len(tokens), color="gray", linestyle="--", label="uniform")
    for i, w in enumerate(weights):
        ax2.text(i, w + 0.01, f"{w:.2f}", ha="center", fontsize=9)
    ax2.set_ylim(0, max(0.55, weights.max() + 0.08))
    ax2.set_title(f"Attention distribution: {name}")
    ax2.set_ylabel("weight")
    ax2.legend()

plt.tight_layout()
plt.show()

print("Reflection prompt:")
print("- Which token receives the largest weight for each query?")
print("- Which tokens stay moderately important rather than dominating completely?")
print("- How does changing only the query change what gets retrieved?")

## Core task 2 — Train a tiny self-attention classifier on a long-range task

Now we move from a hand-computable example to a learnable one.

### Synthetic DGP

Each input is a sequence of integer tokens of fixed length. The label is:

- **1** if the **first** and **last** token match,
- **0** otherwise.

This is intentionally narrow, but it is a nice probe of **long-range retrieval** because the decisive information lives at **two distant positions**.

Our self-attention model will:

1. embed each token,
2. add a learned **positional embedding**,
3. compute **Q, K, V** for the whole sequence,
4. form an attention matrix, and
5. classify from the **last position's attended representation**.

Because the dataset is balanced, **chance accuracy is 50%**.

In [ ]:
# Build the synthetic dataset and a simple self-attention model

VOCAB_SIZE = 10
SEQ_LEN = 24
N_TRAIN = 1200
N_VAL = 300
N_TEST = 300
BATCH_SIZE = 64
D_MODEL = 24
EPOCHS = 8
LR = 3e-3

def make_match_dataset(n, seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE, rng=None):
    rng = np.random.default_rng(SEED) if rng is None else rng
    X = rng.integers(0, vocab_size, size=(n, seq_len))
    half = n // 2

    # Positive examples: first token equals last token
    X[:half, -1] = X[:half, 0]

    # Negative examples: first token and last token must differ
    for i in range(half, n):
        v = X[i, -1]
        while v == X[i, 0]:
            v = rng.integers(0, vocab_size)
        X[i, -1] = v

    y = (X[:, 0] == X[:, -1]).astype(np.int64)
    idx = rng.permutation(n)
    return X[idx], y[idx]

X_train_np, y_train_np = make_match_dataset(N_TRAIN, rng=rng)
X_val_np, y_val_np = make_match_dataset(N_VAL, rng=rng)
X_test_np, y_test_np = make_match_dataset(N_TEST, rng=rng)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_np, dtype=torch.long), torch.tensor(y_train_np, dtype=torch.long)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val_np, dtype=torch.long), torch.tensor(y_val_np, dtype=torch.long)),
    batch_size=256,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test_np, dtype=torch.long), torch.tensor(y_test_np, dtype=torch.long)),
    batch_size=256,
    shuffle=False,
)

class TinySelfAttentionClassifier(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, d_model=D_MODEL, use_positional=True):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_positional = use_positional
        if use_positional:
            self.pos_emb = nn.Embedding(seq_len, d_model)
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.fc = nn.Linear(d_model, 2)
        self.scale = d_model ** 0.5

    def forward(self, x, return_attn=False):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        e = self.token_emb(x)
        if self.use_positional:
            e = e + self.pos_emb(pos)

        Q = self.Wq(e)
        K = self.Wk(e)
        V = self.Wv(e)

        A = torch.softmax((Q @ K.transpose(1, 2)) / self.scale, dim=-1)
        Z = A @ V

        # Use the last position's attended representation for classification.
        logits = self.fc(Z[:, -1, :])
        return (logits, A) if return_attn else logits

attn_model = TinySelfAttentionClassifier()
attn_history = train_classifier(attn_model, train_loader, val_loader, epochs=EPOCHS, lr=LR)
attn_test_acc = accuracy_on_loader(attn_model, test_loader)

print(f"Tiny self-attention parameters: {count_params(attn_model):,}")
print(f"Tiny self-attention test accuracy: {attn_test_acc:.3f}")
attn_history.tail()

In [ ]:
# Plot learning behaviour and inspect learned attention on two test examples

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(attn_history["epoch"], attn_history["train_loss"], marker="o", label="train loss")
ax.set_xlabel("epoch")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Tiny self-attention: training loss")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(attn_history["epoch"], attn_history["val_acc"], marker="o", label="val accuracy")
ax.axhline(0.5, color="gray", linestyle=":", label="chance")
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.set_ylim(0.45, 1.02)
ax.set_title("Tiny self-attention: validation accuracy")
ax.legend()
plt.show()

# Pick one positive and one negative example from the test set
pos_idx = int(np.where(y_test_np == 1)[0][0])
neg_idx = int(np.where(y_test_np == 0)[0][0])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

for ax, idx, title_prefix in [(axes[0], pos_idx, "positive"), (axes[1], neg_idx, "negative")]:
    xb = torch.tensor(X_test_np[idx:idx+1], dtype=torch.long).to(device)
    with torch.no_grad():
        logits, A = attn_model(xb, return_attn=True)
    A0 = A[0].detach().cpu().numpy()
    seq = X_test_np[idx]
    pred = int(logits.argmax(dim=1).item())
    true = int(y_test_np[idx])

    im = ax.imshow(A0, aspect="auto", vmin=0, vmax=1)
    ax.set_title(f"{title_prefix}: true={true}, pred={pred}")
    ax.set_xlabel("attended position j")
    ax.set_ylabel("query position i")

    tick_pos = [0, 5, 10, 15, 20, SEQ_LEN - 1]
    ax.set_xticks(tick_pos)
    ax.set_yticks(tick_pos)

    # Highlight the (last -> first) cell, which is often useful for this task.
    rect = plt.Rectangle((-0.5, SEQ_LEN - 1 - 0.5), 1, 1, fill=False, linewidth=2)
    ax.add_patch(rect)

    ax.text(0.02, 1.02,
            f"first={seq[0]}, last={seq[-1]} | sequence head={seq[:6].tolist()} ...",
            transform=ax.transAxes, fontsize=8)

fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
plt.show()

print("Reflection prompt:")
print("- Does the last row of the attention map place visible weight on the first token?")
print("- Is the pattern identical across examples? Why might that be too strong an expectation?")
print("- Why is this a diagnostic, not a guaranteed explanation?")

## Core task 3 — Compare self-attention with a GRU baseline

The module emphasizes that self-attention can create a **shorter path** between distant positions than recurrence.

To make that visible, we will train a small **GRU** on the same task and compare:

- validation accuracy across epochs,
- final test accuracy, and
- parameter counts.

This is still only a toy task, so do **not** over-generalize from one run. The point is to connect the architectural idea from the video to an observable experiment.

In [ ]:
# Train a GRU baseline on the same task and compare it to self-attention

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, hidden_dim=D_MODEL):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.gru = nn.GRU(d_model, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        e = self.emb(x)
        _, h = self.gru(e)
        return self.fc(h[-1])

gru_model = GRUClassifier()
gru_history = train_classifier(gru_model, train_loader, val_loader, epochs=EPOCHS, lr=LR)
gru_test_acc = accuracy_on_loader(gru_model, test_loader)

summary = pd.DataFrame([
    {"model": "Tiny self-attention", "params": count_params(attn_model), "test_acc": attn_test_acc},
    {"model": "GRU",                 "params": count_params(gru_model),  "test_acc": gru_test_acc},
]).sort_values("test_acc", ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(attn_history["epoch"], attn_history["val_acc"], marker="o", label="self-attention")
ax.plot(gru_history["epoch"],  gru_history["val_acc"],  marker="o", label="GRU")
ax.axhline(0.5, color="gray", linestyle=":", label="chance")
ax.set_xlabel("epoch")
ax.set_ylabel("validation accuracy")
ax.set_ylim(0.45, 1.02)
ax.set_title("Long-range token-matching task: GRU vs self-attention")
ax.legend()
plt.show()

summary

## Stretch tasks (if you finish early)

1. **Remove positional information.**  
   Copy the self-attention model cell, set the positional flag to False, retrain, and compare its accuracy to the positional version. What does this say about the line from the module that **order is not free**?

2. **Change the sequence length.**  
   Try sequence lengths of 12 and 40, regenerate the data, and retrain both models. Which architecture degrades faster as the long-range dependency gets harder?

3. **Inspect the last query row directly.**  
   Instead of plotting the full attention matrix, plot only the final query row as a bar chart for several examples. Is the model consistently using the first position, or does it spread weight across several positions?

4. **Change the pooling choice.**  
   In the self-attention model, replace the last-position representation with the mean of all attended positions and retrain. Does that help or hurt on this specific task?

5. **Be careful with explanations.**  
   Find two examples with the same true label but different attention maps. Why does that matter if someone tries to claim that one attention pattern is *the* explanation?

## Wrap-up

In this mini-lab you:

- computed **scores, softmax weights, and context vectors** on a toy sentence,
- saw that changing the **query** changes what gets retrieved,
- trained a tiny **self-attention** model with **positional embeddings** on a synthetic long-range task,
- inspected learned **attention maps** as diagnostics, and
- compared self-attention to a **GRU** on the same sequence problem.

This mirrors the big idea of M52:

> Instead of forcing the model to compress the past into one rolling summary, attention lets the current computation **retrieve what it needs when it needs it**.

That is why attention was such a consequential step on the way from recurrent sequence models to transformers.